<a href="https://colab.research.google.com/github/claudiohnavarro/CliffWalking/blob/master/HAR_using_CNN__LSTM_Final_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [31]:
import numpy as np
import keras
import _pickle as cPickle

import tensorflow
from numpy import mean
from numpy import std
from numpy import dstack
from pandas import read_csv
from keras.models import Sequential
from keras.layers import Dense, TimeDistributed, Conv1D, MaxPooling1D
from keras.layers import Flatten
from keras.layers import Dropout
from keras.layers import LSTM
from keras.utils import to_categorical
import matplotlib.pyplot as plt
import time
from sklearn import metrics
import csv

In [44]:
def load_file(filepath):
    dataframe = read_csv(filepath, header=None, sep=r'\s+')
    return dataframe.values

In [33]:
# load a list of files and return as a 3d numpy array
def load_group(filenames, prefix=''):
    loaded = list()
    for name in filenames:
        data = load_file(prefix + name)
        loaded.append(data)
    # stack group so that features are the 3rd dimension
    loaded = dstack(loaded)
    return loaded


In [34]:
# load a dataset group, such as train or test
def load_dataset_group(group, prefix=''):
    filepath = prefix + group + '/Inertial Signals/'
    # load all 9 files as a single array
    filenames = list()
    # total acceleration
    filenames += ['total_acc_x_' + group + '.txt', 'total_acc_y_' + group + '.txt', 'total_acc_z_' + group + '.txt']
    # body acceleration
    filenames += ['body_acc_x_' + group + '.txt', 'body_acc_y_' + group + '.txt', 'body_acc_z_' + group + '.txt']
    # body gyroscope
    filenames += ['body_gyro_x_' + group + '.txt', 'body_gyro_y_' + group + '.txt', 'body_gyro_z_' + group + '.txt']
    # load input data
    X = load_group(filenames, filepath)
    # load class output
    y = load_file(prefix + group + '/y_' + group + '.txt')
    return X, y

In [35]:
# load the dataset, returns train and test X and y elements
def load_dataset(prefix=''):
    # load all train
    trainX, trainy = load_dataset_group('train', prefix + '/content/gdrive/My Drive/Colab Notebooks/UCI HAR Dataset/UCI HAR Dataset/')
    print(trainX.shape, trainy.shape)
    # load all test
    testX, testy = load_dataset_group('test', prefix + '/content/gdrive/My Drive/Colab Notebooks/UCI HAR Dataset/UCI HAR Dataset/')
    print(testX.shape, testy.shape)
    # zero-offset class values
    trainy = trainy - 1
    testy = testy - 1
    # one hot encode y
    trainy = to_categorical(trainy)
    testy = to_categorical(testy)

    print(trainX.shape, trainy.shape, testX.shape, testy.shape)
    return trainX, trainy, testX, testy


In [36]:
# fit and evaluate a model
def evaluate_model(trainX, trainy, testX, testy):

    # define model
    verbose, epochs, batch_size = 1, 100,64
    n_timesteps, n_features, n_outputs = trainX.shape[1], trainX.shape[2], trainy.shape[1]
    # reshape data into time steps of sub-sequences
    n_steps, n_length = 4, 32
    trainX = trainX.reshape((trainX.shape[0], n_steps, n_length, n_features))
    testX = testX.reshape((testX.shape[0], n_steps, n_length, n_features))
    # define model
    model = Sequential()
    model.add(TimeDistributed(Conv1D(filters=64, kernel_size=3, activation='relu'), input_shape=(None, n_length, n_features)))
    model.add(TimeDistributed(Conv1D(filters=64, kernel_size=3, activation='relu')))
    model.add(TimeDistributed(Dropout(0.5)))
    model.add(TimeDistributed(MaxPooling1D(pool_size=2)))
    model.add(TimeDistributed(Flatten()))
    model.add(LSTM(100))
    model.add(Dropout(0.5))
    model.add(Dense(100, activation='relu'))
    model.add(Dense(n_outputs, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.summary()
     # fit network
   # ts1 = time.time()
   # print ("time1",ts1)
    history=model.fit(trainX, trainy, epochs=epochs, batch_size=batch_size, verbose=verbose, validation_data=(testX, testy))

   # ts2 = time.time()
   # print ("time2",ts2)
   # print("time2-time1",(ts2-ts1))
       # test model
    #ts3 = time.time()
    #print ("time3",ts3)
    test_model(model, testX, verbose, batch_size, n_outputs)
    _, accuracy = model.evaluate(testX, testy, batch_size=batch_size, verbose=verbose)
    #ts4 = time.time()
    #print ("time4",ts4)
    #print("time4-time3",(ts4-ts3))
    return accuracy,history


In [37]:
# fit and evaluate a model
def test_model(model, testX, verbose, batch_size, n_outputs):

    prediction_list = model.predict(testX, batch_size=batch_size, steps=None, verbose=verbose)
    predictions_transformed = np.eye(n_outputs, dtype=int)[np.argmax(prediction_list, axis=1)]

    np.savetxt('predictions.txt', prediction_list)
    np.savetxt('predictions_trans.txt', (np.argmax(predictions_transformed, axis=1)))

In [38]:
def plot_accuracy(history):
      acc = history.history['acc']
      val_acc = history.history['val_acc']
      loss = history.history['loss']
      val_loss = history.history['val_loss']
      epochs = range(1, len(acc) + 1)
      plt.plot(epochs, acc, 'bo', label='Training acc')
      plt.plot(epochs, val_acc, 'orange', label='Validation acc')
      plt.title('Training and validation accuracy')
      plt.legend()
      plt.figure()

In [39]:
def plot_loss(history):
      acc = history.history['acc']
      val_acc = history.history['val_acc']
      loss = history.history['loss']
      val_loss = history.history['val_loss']
      epochs = range(1, len(acc) + 1)
      plt.plot(epochs, loss, 'bo', label='Training loss')
      plt.plot(epochs, val_loss, 'orange', label='Validation loss')
      plt.title('Training and validation loss')
      plt.legend()
      plt.figure()

In [40]:
# summarize scores
def summarize_results(scores):
    print(scores)
    m, s = mean(scores), std(scores)
    print('Accuracy: %.3f%% (+/-%.3f)' % (m, s))

In [45]:
def plot_predictions():
    x = list(range(1, 101))
    actual = read_csv("/content/gdrive/My Drive/Colab Notebooks/UCI HAR Dataset/UCI HAR Dataset/test/y_test.txt", nrows=100, header=None, sep=r'\s+')
    predicted = read_csv("predictions_trans.txt", nrows=100, header=None, sep=r'\s+')
    predicted = ([x+1 for x in np.array(predicted)])
    plt.plot(x, actual)
    plt.plot(x, predicted, color='r')
    plt.show()

    with open('/content/gdrive/My Drive/Colab Notebooks/UCI HAR Dataset/UCI HAR Dataset/test/y_test.txt', newline='') as csvfile:
       actualok = list(csv.reader(csvfile))

    with open('predictions_trans.txt', newline='') as csvfile:
       predictedok = list(csv.reader(csvfile))


    print("Confusion Matrix:")
    confusion_matrix = metrics.confusion_matrix(actualok, predictedok)
    print(confusion_matrix)
    normalised_confusion_matrix = np.array(confusion_matrix, dtype=np.float32)/np.sum(confusion_matrix)*100
    # Plot Results:
    width = 8
    height = 8

    plt.figure(figsize=(width, height))
    plt.imshow(
        normalised_confusion_matrix,
        interpolation='nearest',
        cmap=plt.cm.rainbow
    )
    plt.title("Confusion matrix \n(normalised to % of total test data)")


    print(normalised_confusion_matrix)

    LABELS = [
        "WALKING",
        "WALKING_UPSTAIRS",
        "WALKING_DOWNSTAIRS",
        "SITTING",
        "STANDING",
        "LAYING"
    ]
    plt.colorbar()
    tick_marks = np.arange(6)
    plt.xticks(tick_marks, LABELS, rotation=90)
    plt.yticks(tick_marks, LABELS)
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

In [42]:
def run_experiment(repeats=5):
    # load data
    trainX, trainy, testX, testy = load_dataset()
    #repeat experiment
    scores=list()
    for r in range(repeats):
        score,history = evaluate_model(trainX, trainy, testX, testy)
        score=score*100.0
        print("Accuracy : ", score)
        plot_predictions()
        plot_accuracy(history)
        plot_loss(history)

In [43]:
# run the experiment
run_experiment()

/tmp/ipykernel_7269/776785211.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  dataframe = read_csv(filepath, header=None, delim_whitespace=True)


FileNotFoundError: [Errno 2] No such file or directory: '/content/gdrive/My Drive/Colab Notebooks/UCI HAR Dataset/UCI HAR Dataset/train/Inertial Signals/total_acc_x_train.txt'

In [46]:
import os

# Definimos la ruta base de la carpeta
path = '/content/gdrive/My Drive/Colab Notebooks/UCI HAR Dataset/'

try:
    # Listamos el contenido de la carpeta
    print("Contenido de la carpeta principal:")
    print(os.listdir(path))
except FileNotFoundError:
    print(f"La ruta '{path}' no existe. Por favor, verifica el nombre exacto de tus carpetas en Google Drive.")

Contenido de la carpeta principal:
La ruta '/content/gdrive/My Drive/Colab Notebooks/UCI HAR Dataset/' no existe. Por favor, verifica el nombre exacto de tus carpetas en Google Drive.


In [47]:
import os

def buscar_carpeta(nombre_carpeta, ruta_busqueda):
    coincidencias = []
    for raiz, directorios, archivos in os.walk(ruta_busqueda):
        if nombre_carpeta in directorios:
            coincidencias.append(os.path.join(raiz, nombre_carpeta))
    return coincidencias

# Buscaremos en todo el directorio de Google Drive
ruta_drive = '/content/gdrive/My Drive/'
if os.path.exists(ruta_drive):
    print("Buscando 'UCI HAR Dataset' en Google Drive...")
    resultados = buscar_carpeta('UCI HAR Dataset', ruta_drive)
    if resultados:
        print("\nSe encontraron las siguientes rutas:")
        for r in resultados:
            print(r)
    else:
        print("\nNo se encontró ninguna carpeta con el nombre 'UCI HAR Dataset' en tu Drive.")
else:
    print("Google Drive no parece estar montado en '/content/gdrive/My Drive/'. Revisa la conexión de Drive.")

Buscando 'UCI HAR Dataset' en Google Drive...

No se encontró ninguna carpeta con el nombre 'UCI HAR Dataset' en tu Drive.


In [ ]:
import pandas as pd
df = pd.read_csv('predictions_trans.txt', sep=" ", header=None, names=["Prediction"])
temp_dict={0:'Walking',
           1:'Walking-upstairs',
           2:'Walking-downstairs',
           3:'Sitting',
           4:'Standing',
           5:'Laying'}

df['Prediction']=df.Prediction.map(temp_dict)
df

In [ ]:
df.to_csv(r'activity_predictions.txt', header=None, index=None, sep='\t', mode='a')